Set GPU

In [ ]:
import matplotlib
import torch

is_ipython = 'inline' in matplotlib.get_backend()
device = torch.device(
     "cuda" if torch.cuda.is_available() else
     "mps" if torch.backends.mps.is_available() else
     "cpu"
)
print("device:", device)


device: cuda


Run RL

In [ ]:
#########
# User-define values (Baseline)

EPISODE = 300
BATCH_SIZE = 128
LR = 1e-4 
MAX_STEPS = 100

config = {
    "canvas": [10,10],
    "inventory": {
        "1": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "2": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "3": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "4": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "5": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "6": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "7": [(0,0),(3,0),(3,3),(0,3),(0,0)],
        "8": [(0,0),(4,0),(4,4),(0,4),(0,0)],
        "9": [(0,0),(3,0),(3,5),(0,5),(0,0)],
        "10": [(0,0),(2,0),(2,4),(0,4),(0,0)],
        "11": [(0,0),(1,0),(1,5),(0,5),(0,0)],
        "12": [(0,0),(1,0),(1,4),(0,4),(0,0)],
        "13": [(0,0),(1,0),(1,3),(0,3),(0,0)],
        "14": [(0,0),(1,0),(1,3),(0,3),(0,0)],
        "15": [(0,0),(2,0),(2,3),(1,3),(1,4),(0,4),(0,0)]
    },
    "weight": [2,2,1],
    "max_number_of_stocks": 15,
    "max_step": MAX_STEPS
}

#########

from dqn import main
from envs import reuse_r as renv

EPS_START = 0.2     # 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TAU = 0.005
DECAY_RATE = 0.99

# Run
env = renv.ReuseTetris(config=config)
main.main(
     env=env,
     EPISODE = EPISODE, BATCH_SIZE = BATCH_SIZE,
     EPS_START = EPS_START, EPS_END = EPS_END, EPS_DECAY = EPS_DECAY,
     TAU = TAU, LR = LR, DECAY_RATE = DECAY_RATE,
     MAX_STEPS = MAX_STEPS
)


Sample RL result analysis

In [ ]:
# Read the result JSON file

import json

file_name = r"./result/result_YYYYMMDD-TTTT.json"

with open(file_name, 'r', encoding='utf-8') as f:
    data = json.load(f)
data = json.loads(data)
default_ep = list(data.keys())

print(default_ep)

dict_keys(['stock_queue', 'step1', 'step2', 'step3', 'step4', 'step5', 'step6', 'step7', 'step8', 'step9', 'step10', 'step11', 'step12', 'step13', 'step14', 'step15', 'step16', 'step17', 'step18', 'step19', 'step20', 'step21', 'step22', 'step23', 'step24', 'step25', 'step26', 'step27', 'step28', 'step29', 'step30', 'step31', 'step32', 'step33', 'step34', 'step35', 'step36', 'step37', 'step38', 'step39', 'step40', 'step41', 'step42', 'step43', 'step44', 'step45', 'step46', 'step47', 'step48', 'step49', 'step50', 'step51', 'step52', 'step53', 'step54', 'step55', 'step56', 'step57', 'step58', 'step59', 'step60', 'step61', 'step62', 'step63', 'step64', 'step65', 'step66', 'step67', 'step68', 'step69', 'step70', 'step71', 'step72', 'step73', 'step74', 'step75', 'step76', 'step77', 'step78', 'step79', 'step80', 'step81', 'step82', 'step83', 'step84', 'step85', 'step86', 'step87', 'step88', 'step89', 'step90', 'step91', 'step92', 'step93', 'step94', 'step95', 'step96', 'step97', 'step98', 'st

In [ ]:
# Retrieve state images of a chosen EP

ep = 0   # Input EP of interest

# prep Polygons

import shapely
from shapely import Polygon
from shapely.geometry import Polygon, LineString
from shapely.ops import split

import matplotlib
import matplotlib.pyplot as plt
import io
from PIL import Image

pol_canvas_diff = Polygon([(0,0),(10,0),(10,10),(0,10),(0,0)])
pol_union = Polygon()
inventory = {}
for id, stock_pts in config["inventory"].items():
    inventory[id] = Polygon(stock_pts)

best_ep = data[f'episode_{ep}']
stock_used = []

# Plot steps of the EP
steps = range(1, MAX_STEPS+1)
for s in steps:    
    # Retrieve info
    step_info = best_ep[f'step{s}']
    stock_id = step_info['stock_id']
    action = step_info['action']

    # Processing
    polygon_cur = inventory[stock_id]

    if action == 0:     # Placement
        if shapely.covers(pol_canvas_diff, polygon_cur):
            stock_used.append(stock_id)
            pol_canvas_diff = shapely.difference(pol_canvas_diff, polygon_cur)

    elif action == 1:   # Next stock on queue
        continue

    elif action == 2 or action == 3:   # Cut

        if action == 2:     # Cut parallel to X-axis
            temp = list(set([y for _, y in list(polygon_cur.exterior.coords)]))
            if len(temp) == 2 and temp[1]-temp[0] > 1:
                y2_stock_cur = temp[1]-1
            else:
                y2_stock_cur = temp[1]
            cutting_line = LineString([(-100,y2_stock_cur), (100,y2_stock_cur)])

        else:               # Cut parallel to Y-axis
            temp = list(set([x for x, _ in list(polygon_cur.exterior.coords)]))
            if len(temp) == 2 and temp[1]-temp[0] > 1:
                x2_stock_cur = temp[1]-1
            else:
                x2_stock_cur = temp[1]
            cutting_line = LineString([(x2_stock_cur,-100), (x2_stock_cur,100)])

        pol_stocks_added = sorted(split(polygon_cur, cutting_line).geoms, key=lambda x: x.area)
        polygon_cur = pol_stocks_added.pop()
            
        # Update Env
        inventory[stock_id] = polygon_cur
        for pol_temp in pol_stocks_added:
            pol_temp = shapely.affinity.translate(pol_temp,-pol_temp.bounds[0],-pol_temp.bounds[1]) # Move back to (0,0)
            max_id += 1
            inventory[str(max_id)] = pol_temp

    elif action == 4:   # Rotation
        temp = shapely.affinity.rotate(polygon_cur, 90)
        pol_stock_cur = shapely.affinity.translate(
            temp,
            -temp.bounds[0],
            -temp.bounds[1]
        )
        inventory[stock_id] = polygon_cur

    elif action < 7:   # Translate X
        polygon_cur = shapely.affinity.translate(polygon_cur,(-1)**(5-action),0)
        inventory[stock_id] = polygon_cur
        
    else:
        polygon_cur = shapely.affinity.translate(polygon_cur,0,(-1)**(7-action))
        inventory[stock_id] = polygon_cur

# Draw
fig = plt.figure(1, figsize=(5,5))
ax = fig.add_subplot(111, xlim=(0,10), ylim=(0,10), aspect="equal")

for id_used in stock_used:
    polygon = inventory[id_used]
    ax.fill(*polygon.exterior.xy, color="blue", alpha=0.4, zorder=1)
    ## Add stock ID
    # x, y = tuple(polygon.representative_point().coords)[0]
    # ax.text(x, y, f'{id_used}')
# Remove XY ticks
ax.set_xticks([])
ax.set_yticks([])

# Export
buf = io.BytesIO()
plt.savefig(buf, format='png')
buf.seek(0)
image = Image.open(buf).convert('RGB')
image.save(f"./result/ep {ep}_Final layout22.jpg")
plt.clf()
